# Import Gapminder data

https://www.gapminder.org/data/

In [ ]:
# pandas to the rescue
import pandas as pd

In [ ]:
# read CSV files
pop = pd.read_csv('pop.csv')
gdp = pd.read_csv('gdp.csv')
lex = pd.read_csv('lex.csv')

In [ ]:
# check the data
pop.head()

Oh no, the data has "k"'s and "M"'s in the numbers! We need to clean it up.

In [ ]:
# convert text numbers to numeric numbers
def _parse_num(v):
    if pd.isna(v):
        return pd.NA
    if isinstance(v, (int, float)):
        return int(v)
    s = str(v).strip()
    if not s:
        return pd.PA
    mult = 1
    if s.endswith('k'):
        mult = 1_000
        s = s[:-1]
    elif s.endswith('M'):
        mult = 1_000_000
        s = s[:-1]
    try:
        return int(float(s) * mult)
    except ValueError:
        return pd.NA

for _df in (pop, gdp, lex):
    value_cols = [c for c in _df.columns if c != 'country']
    _df[value_cols] = _df[value_cols].applymap(_parse_num).astype('Int64')

In [ ]:
# check that it worked
pop.head()


We want to convert the data from "wide" to "long" format

In [ ]:
# Pivot and rename columns for each dataframe

pop_long = pop.melt(id_vars='country', var_name='Year', value_name='Population')
pop_long = pop_long.rename(columns={'country': 'Country'})

gdp_long = gdp.melt(id_vars='country', var_name='Year', value_name='GDP')
gdp_long = gdp_long.rename(columns={'country': 'Country'})

lex_long = lex.melt(id_vars='country', var_name='Year', value_name='Life Expectancy')
lex_long = lex_long.rename(columns={'country': 'Country'})

In [ ]:
pop_long.head()

In [ ]:
# Merge the three dataframes on 'Country' and 'Year'
hans = pop_long.merge(gdp_long, on=['Country', 'Year'], how='outer') \
               .merge(lex_long, on=['Country', 'Year'], how='outer')

In [ ]:
hans.head()

In [ ]:
countries = pd.read_csv('countries.csv')

In [ ]:
# Merge 'region' and 'sub-region' from countries into hans by matching hans['Country'] to countries['name']
hans = hans.merge(
    countries[['name', 'region', 'sub-region']],
    left_on='Country',
    right_on='name',
    how='left'
).drop(columns=['name'])

In [ ]:
hans.head()

In [ ]:
# export to CSV
hans.to_csv('hans.csv', index=False)